# C7-cnn-transfer — Practice p10 — Solution

**Type:** constrained coding · **Difficulty:** core · **Concepts:** layer-freezing, nn-module, requires-grad, parameter-counting, cnn-training

The qualified-name prefix test separates the model before the optimizer is constructed. There are two useful currencies: eight frozen parameter tensors, but 333 frozen scalar entries. Adam receives only the four trainable tensors owned by `layer3` and `fc`; the final audit checks object identity, gradients, movement, loss, and exact immobility of the frozen tensors.

In [ ]:
import torch
from torch import nn

torch.set_default_dtype(torch.float64)
SEED = 20260804
torch.manual_seed(SEED)

class TinyTransferCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 3, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(3)
        self.layer1 = nn.Sequential(nn.Conv2d(3, 4, 3, padding=1), nn.ReLU())
        self.layer2 = nn.Sequential(nn.Conv2d(4, 5, 3, padding=1), nn.ReLU())
        self.layer3 = nn.Sequential(nn.Conv2d(5, 6, 3, padding=1), nn.ReLU())
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(6, 3)

    def forward(self, x):
        x = torch.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return self.fc(torch.flatten(self.pool(x), 1))

generator = torch.Generator(device="cpu").manual_seed(SEED)
X = 0.05 * torch.randn(18, 1, 6, 6, generator=generator)
y = torch.arange(18, dtype=torch.long) % 3
X[y == 0, :, :, 1:3] += 1.0
X[y == 1, :, 3:5, :] += 1.0
diagonal = torch.arange(6)
class_two = X[y == 2].clone()
class_two[:, :, diagonal, diagonal] += 1.0
X[y == 2] = class_two
X[y == 0] -= 0.8
X[y == 2] += 0.8
model = TinyTransferCNN()
criterion = nn.CrossEntropyLoss()

## Part I — selective freezing and two-currency audit

In [ ]:
frozen_prefixes = ("conv1", "bn1", "layer1", "layer2")  # PLAN017_MUTATION_TARGET: C7-p10-frozen-update
for name, parameter in model.named_parameters():
    parameter.requires_grad_(not name.startswith(frozen_prefixes))

frozen_parameters = [parameter for parameter in model.parameters() if not parameter.requires_grad]
trainable_parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
n_frozen_tensors = len(frozen_parameters)
n_frozen_scalars = sum(parameter.numel() for parameter in frozen_parameters)
n_trainable_scalars = sum(parameter.numel() for parameter in trainable_parameters)
split_ok = n_frozen_scalars + n_trainable_scalars == sum(
    parameter.numel() for parameter in model.parameters()
)
trainable_tops = sorted(
    {name.split(".", maxsplit=1)[0] for name, parameter in model.named_parameters() if parameter.requires_grad}
)

## Part II — train only what the audit permits

In [ ]:
parameter_before = {
    name: parameter.detach().clone() for name, parameter in model.named_parameters()
}
optimizer = torch.optim.Adam(trainable_parameters, lr=0.05)
loss_history = []
for _ in range(18):
    optimizer.zero_grad(set_to_none=True)
    logits = model(X)
    loss = criterion(logits, y)
    loss_history.append(float(loss.detach()))
    loss.backward()
    optimizer.step()

trainable_names = sorted(
    name for name, parameter in model.named_parameters() if parameter.requires_grad
)
frozen_names = sorted(
    name for name, parameter in model.named_parameters() if not parameter.requires_grad
)
optimizer_parameter_objects = [
    parameter for group in optimizer.param_groups for parameter in group["params"]
]
optimizer_parameter_ids = {id(parameter) for parameter in optimizer_parameter_objects}
optimizer_owns_exactly_trainable = (
    optimizer_parameter_ids == {id(parameter) for parameter in trainable_parameters}
    and len(optimizer_parameter_objects) == len(trainable_parameters)
)
gradient_names = sorted(
    name for name, parameter in model.named_parameters() if parameter.grad is not None
)
moved_trainable_names = sorted(
    name
    for name, parameter in model.named_parameters()
    if parameter.requires_grad and not torch.equal(parameter.detach(), parameter_before[name])
)
frozen_bitwise_unchanged = all(
    torch.equal(parameter.detach(), parameter_before[name])
    for name, parameter in model.named_parameters()
    if not parameter.requires_grad
)
training_certificate = bool(
    len(loss_history) == 18
    and torch.isfinite(torch.tensor(loss_history)).all()
    and loss_history[-1] <= 0.80 * loss_history[0]
    and optimizer_owns_exactly_trainable
    and gradient_names == trainable_names
    and len(moved_trainable_names) > 0
    and frozen_bitwise_unchanged
)

### Answer check

In [ ]:
# PLAN017_ANSWER_CHECK: C7-p10-freezing
expected_frozen_names = sorted(
    name
    for name, _ in model.named_parameters()
    if name.startswith(("conv1", "bn1", "layer1", "layer2"))
)
expected_trainable_names = sorted(
    name for name, _ in model.named_parameters() if name not in expected_frozen_names
)
actual_frozen_names = sorted(
    name for name, parameter in model.named_parameters() if not parameter.requires_grad
)
actual_trainable_names = sorted(
    name for name, parameter in model.named_parameters() if parameter.requires_grad
)
assert actual_frozen_names == expected_frozen_names
assert actual_trainable_names == expected_trainable_names
assert (n_frozen_tensors, n_frozen_scalars, n_trainable_scalars) == (8, 333, 297)
assert split_ok and n_frozen_scalars + n_trainable_scalars == 630
assert trainable_tops == ["fc", "layer3"]
assert optimizer_parameter_ids == {
    id(parameter) for name, parameter in model.named_parameters() if name in expected_trainable_names
}
assert len(optimizer_parameter_objects) == len(expected_trainable_names)
assert optimizer_owns_exactly_trainable
assert gradient_names == expected_trainable_names
assert moved_trainable_names
assert set(moved_trainable_names).issubset(expected_trainable_names)
assert all(
    torch.equal(parameter.detach(), parameter_before[name])
    for name, parameter in model.named_parameters()
    if name in expected_frozen_names
)
assert len(loss_history) == 18
assert torch.isfinite(torch.tensor(loss_history)).all()
assert loss_history[-1] <= 0.80 * loss_history[0]
assert frozen_bitwise_unchanged and training_certificate